In [1]:
!pip install transformers sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 22.9 MB/s eta 0:00:00


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer
import faiss

knowledge_base = [
    "O Projeto Alpha é um sistema de IA que utiliza aprendizado por reforço para otimizar rotas logísticas.",
    "A reunião de status do Projeto Alpha acontecerá na próxima quinta-feira às 14h, na sala Júpiter.",
    "A equipe de desenvolvimento do Projeto Alpha é composta por João, Maria e Pedro, e o líder é a Dra. Ana.",
    "O framework principal utilizado no Projeto Alpha para inferência é o PyTorch, otimizado com a biblioteca ONNX.",
    "A fase Beta do Projeto Alpha foi concluída com sucesso no dia 15 de novembro.",
]

embedding_model_name = 'all-MiniLM-L6-v2'
embedding_model = SentenceTransformer(embedding_model_name)

llm_name = 'google/flan-t5-small'
tokenizer = AutoTokenizer.from_pretrained(llm_name)
llm_model = AutoModelForSeq2SeqLM.from_pretrained(llm_name)


print(f"Modelos carregados: Embeddings ({embedding_model_name}) e LLM ({llm_name}).")

print("Criando embeddings da base de conhecimento...")

embeddings = embedding_model.encode(knowledge_base, convert_to_tensor=False)
d = embeddings.shape[1]
index = faiss.IndexFlatL2(d)
index.add(embeddings)

print(f"Base de conhecimento indexada com {len(knowledge_base)} vetores.")

def rag_query(user_query, top_k=2):
    """Executa o pipeline RAG: Busca o contexto e gera a resposta."""
    query_embedding = embedding_model.encode(user_query, convert_to_tensor=False).reshape(1, -1)

    distances, indices = index.search(query_embedding, top_k)

    context_chunks = [knowledge_base[i] for i in indices[0]]
    retrieved_context = " ".join(context_chunks)

    print("\n--- Contexto Recuperado ---")
    print(retrieved_context)
    print("---------------------------\n")

    prompt = f"Use o seguinte contexto para responder à pergunta: '{retrieved_context}'. Pergunta: '{user_query}'"

    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
    outputs = llm_model.generate(**inputs, max_length=100)
    final_answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return final_answer

user_question_1 = "Quem faz parte da equipe de desenvolvimento do Projeto Alpha?"
answer_1 = rag_query(user_question_1)

print(f"Pergunta: {user_question_1}")
print(f"Resposta Final: **{answer_1}**\n")

user_question_2 = "Onde e quando será a próxima reunião?"
answer_2 = rag_query(user_question_2)

print(f"Pergunta: {user_question_2}")
print(f"Resposta Final: **{answer_2}**")

Modelos carregados: Embeddings (all-MiniLM-L6-v2) e LLM (google/flan-t5-small).
Criando embeddings da base de conhecimento...
Base de conhecimento indexada com 5 vetores.

--- Contexto Recuperado ---
O Projeto Alpha é um sistema de IA que utiliza aprendizado por reforço para otimizar rotas logísticas. A fase Beta do Projeto Alpha foi concluída com sucesso no dia 15 de novembro.
---------------------------

Pergunta: Quem faz parte da equipe de desenvolvimento do Projeto Alpha?
Resposta Final: **What is the name of the IA system?**


--- Contexto Recuperado ---
A reunião de status do Projeto Alpha acontecerá na próxima quinta-feira às 14h, na sala Júpiter. O Projeto Alpha é um sistema de IA que utiliza aprendizado por reforço para otimizar rotas logísticas.
---------------------------

Pergunta: Onde e quando será a próxima reunião?
Resposta Final: **When will the first meeting be held?**
